# Day 045 Solution — Data Pipelines

extract, validate_record, transform_record, load, run_pipeline. Self-contained: CSV_SOURCE defined inline, in-memory SQLite via SQLAlchemy.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import csv
import io
from sqlalchemy import create_engine, String, Float, select, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class Sale(Base):
    __tablename__ = 'sales'
    id:       Mapped[int]   = mapped_column(primary_key=True)
    date:     Mapped[str]   = mapped_column(String(20))
    product:  Mapped[str]   = mapped_column(String(100))
    category: Mapped[str]   = mapped_column(String(50))
    amount:   Mapped[float] = mapped_column()
    region:   Mapped[str]   = mapped_column(String(50))

    def __repr__(self):
        return f'Sale(id={self.id}, product={self.product!r}, amount={self.amount})'


def setup_engine(url='sqlite:///:memory:'):
    engine = create_engine(
        url,
        connect_args={'check_same_thread': False},
        poolclass=StaticPool,
    )
    Base.metadata.create_all(engine)
    return engine


def extract(csv_text: str) -> list:
    reader = csv.DictReader(io.StringIO(csv_text))
    return list(reader)


def validate_record(record: dict) -> bool:
    required = ['date', 'product', 'amount']
    for field in required:
        if not record.get(field, '').strip():
            return False
    try:
        float(record['amount'])
    except (ValueError, TypeError):
        return False
    return True


def transform_record(record: dict) -> dict:
    return {
        'date':     record['date'].strip(),
        'product':  record['product'].strip(),
        'category': record.get('category', '').strip(),
        'amount':   round(float(record['amount']), 2),
        'region':   record.get('region', '').strip().title(),
    }


def load(session, records: list) -> int:
    sales = [Sale(**r) for r in records]
    session.add_all(sales)
    session.commit()
    return len(sales)


def run_pipeline(csv_text: str, session) -> dict:
    raw_records  = extract(csv_text)
    valid        = [r for r in raw_records if validate_record(r)]
    transformed  = [transform_record(r) for r in valid]
    loaded_count = load(session, transformed)
    return {
        'extracted': len(raw_records),
        'loaded':    loaded_count,
        'skipped':   len(raw_records) - loaded_count,
    }


CSV_SOURCE = (
    'date,product,category,amount,region\n'
    '2024-01-15,Laptop,Electronics,999.99,East\n'
    '2024-01-16,Headphones,Electronics,149.99,West\n'
    '2024-01-17,Desk Chair,Furniture,349.00,East\n'
    '2024-01-18,,Furniture,199.00,North\n'
    '2024-01-19,Pen Set,Stationery,twelve,South\n'
    '2024-01-20,Monitor,Electronics,599.99,West\n'
    '2024-01-21,Keyboard,Electronics,79.99,East\n'
    '2024-01-22,Webcam,Electronics,,North\n'
    '2024-01-23,Lamp,Furniture,45.99,South\n'
    '2024-01-24,Notebook,Stationery,8.99,West\n'
)

## Step 1 — Engine

In [ ]:
engine  = setup_engine()
session = Session(engine)
from sqlalchemy import inspect as sa_inspect
assert 'sales' in sa_inspect(engine).get_table_names()
print('Engine ready, sales table created.')

## Step 2 — extract

In [ ]:
records = extract(CSV_SOURCE)
assert len(records) == 10
assert all(isinstance(r, dict) for r in records)
assert records[0]['amount'] == '999.99'   # string, not float
print(f'Extracted {len(records)} records')

## Step 3 — validate_record

In [ ]:
valid_r = {'date':'2024-01-15','product':'Laptop','category':'Electronics','amount':'999.99','region':'East'}
assert validate_record(valid_r) is True
assert validate_record({**valid_r, 'product': ''}) is False
assert validate_record({**valid_r, 'amount': 'twelve'}) is False
assert validate_record({**valid_r, 'amount': ''}) is False
valid_count = sum(1 for r in records if validate_record(r))
assert valid_count == 7, f'expected 7 valid, got {valid_count}'
print(f'Valid: {valid_count} / {len(records)}')

## Step 4 — transform_record

In [ ]:
raw_with_spaces = {'date': ' 2024-01-15 ', 'product': ' Laptop ',
                   'category': 'Electronics', 'amount': '999.99', 'region': 'east'}
t = transform_record(raw_with_spaces)
assert isinstance(t['amount'], float)
assert t['product'] == 'Laptop'
assert t['region'] == 'East'
print(f'Transformed: {t}')

## Step 5 — load

In [ ]:
valid_records   = [r for r in records if validate_record(r)]
transformed     = [transform_record(r) for r in valid_records]
loaded_count    = load(session, transformed)
assert loaded_count == 7
db_count = len(session.execute(select(Sale)).scalars().all())
assert db_count == 7
print(f'Loaded {loaded_count} rows into DB')

## Step 6 — run_pipeline (fresh session)

In [ ]:
engine2  = setup_engine()
session2 = Session(engine2)
stats = run_pipeline(CSV_SOURCE, session2)
assert stats['extracted'] == 10
assert stats['loaded']    == 7
assert stats['skipped']   == 3
print(f'Pipeline stats: {stats}')

# Region check — all title-cased
regions = [s.region for s in session2.execute(select(Sale)).scalars().all()]
assert all(r == r.title() for r in regions if r)
print(f'Regions: {sorted(set(regions))}')

# Category summary
rows = session2.execute(
    text('SELECT category, COUNT(*) as c, ROUND(SUM(amount),2) as t '
         'FROM sales GROUP BY category ORDER BY t DESC')
).mappings().all()
for row in rows:
    print(f'  {row["category"]}: {row["c"]} sales, ${row["t"]:.2f}')

session2.close()
print('\nAll solution checks passed.')